# 🎯 Few-Shot Prompting: Classification Accuracy Comparison

## Motivation

**Few-shot learning** (a.k.a. in-context learning) is the ability of LLMs to
perform a task after seeing only a handful of examples — *without* any gradient
updates or fine-tuning. By prepending labeled examples to the prompt, we:

1. **Calibrate** the model's output format and style
2. **Demonstrate** the desired reasoning or classification logic
3. **Reduce ambiguity** — the model infers the task boundary from examples

In this notebook, we measure how classification accuracy scales with the number
of few-shot examples (0, 2, 4, and 6 shots) on a **customer support ticket
classification task** — routing tickets into **Billing**, **Technical**,
**Account**, or **General** categories.

In [ ]:
%matplotlib inline
import os
import sys
import re
from dataclasses import dataclass, field

# Add project root to path (notebook runs from experiments/ dir)
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "../..")))

from prompts.template import PromptTemplate
from function_caller.config import GenerationConfig

# Auto-detect API availability
API_KEY = os.environ.get("OPENAI_API_KEY", "")
LIVE_API = bool(API_KEY)
print(f"🔑 OPENAI_API_KEY {'found' if LIVE_API else 'NOT found — using mock client'}")

## Mock LLM Client (Fallback for Demo Without API Key)

When no `OPENAI_API_KEY` is set, we use a deterministic mock that simulates
how accuracy improves with more few-shot examples. The mock returns correct
labels with a probability that increases as `n_shots` grows.

In [ ]:
class MockLLMClient:
    """Deterministic mock LLM that models few-shot accuracy improvement.

    Simulates the diminishing-returns pattern: more shots yield higher accuracy,
    but the marginal gain decreases. Uses a per-case lookup so results are
    reproducible across runs.
    """

    def __init__(self):
        self.model_name = "mock-gpt"
        # Per-case correctness thresholds: (correct_if_shots >= N, correct_label)
        self._case_thresholds = [
            (2, "Billing"),      # needs 2+ shots
            (2, "Technical"),
            (6, "Billing"),      # needs 6 shots (hard case)
            (2, "Account"),
            (0, "Technical"),    # easy — always correct
            (4, "Technical"),    # needs 4+ shots
            (2, "Account"),
            (4, "General"),
            (4, "Billing"),
            (0, "General"),      # easy
            (2, "Technical"),
            (0, "Account"),      # easy
            (6, "General"),      # hard
            (4, "Billing"),
            (2, "Account"),
            (6, "Technical"),    # hard
            (0, "Billing"),      # easy
            (4, "General"),
            (2, "Technical"),
            (0, "Account"),      # easy
        ]
        # Wrong-label fallback per case (simulates common confusion)
        self._wrong_labels = [
            "Technical", "Account", "General", "Billing",
            "Billing", "Account", "Billing", "Account",
            "Technical", "Technical", "Billing", "Billing",
            "Account", "General", "Technical", "Billing",
            "General", "Technical", "General", "General",
        ]

    def chat_completion(self, messages: list[dict], **kwargs) -> dict:
        prompt = messages[-1]["content"] if messages else ""
        # Extract case index from prompt
        match = re.search(r"Ticket:\n(.+?)\n\n", prompt)
        ticket_text = match.group(1) if match else ""
        case_idx = hash(ticket_text) % len(self._case_thresholds)
        # Extract n_shots from prompt
        shots_match = re.search(r"You are a customer support ticket classifier", prompt)
        example_count = prompt.count("Example:")

        threshold, correct_label = self._case_thresholds[case_idx]
        if example_count >= threshold:
            label = correct_label
        else:
            label = self._wrong_labels[case_idx]

        return {
            "content": label,
            "role": "assistant",
            "model": self.model_name,
            "usage": {"prompt_tokens": len(prompt) // 4, "completion_tokens": len(label) // 4, "total_tokens": (len(prompt) + len(label)) // 4},
            "finish_reason": "stop",
            "tool_calls": [],
        }


def get_client():
    """Return OpenAIClient if API key is set, otherwise MockLLMClient."""
    if LIVE_API:
        from llm_client.openai_client import OpenAIClient
        return OpenAIClient(api_key=API_KEY, model_name="gpt-4o")
    return MockLLMClient()


client = get_client()
print(f"📡 Using: {client.model_name}")

## Classification Task & Labels

We classify customer support tickets into one of four categories:

| Label | Description |
|-------|-------------|
| **Billing** | Charges, invoices, refunds, payment issues, plan pricing |
| **Technical** | Bugs, errors, performance, integrations, API issues |
| **Account** | Login, password reset, profile settings, permissions |
| **General** | Product questions, feature requests, feedback, other |

In [ ]:
CATEGORIES = ["Billing", "Technical", "Account", "General"]

@dataclass
class Ticket:
    """A customer support ticket with its text and expected label."""
    text: str
    expected: str  # one of: Billing, Technical, Account, General


# ── Test Set: 20 diverse tickets ──────────────────────────────────────────
TEST_SET: list[Ticket] = [
    Ticket("I was charged twice for my subscription this month. Can I get a refund?", "Billing"),
    Ticket("The app crashes whenever I try to upload a file larger than 10MB.", "Technical"),
    Ticket("I forgot my password and the reset email isn't arriving.", "Account"),
    Ticket("What's the difference between the Pro and Enterprise plans?", "Billing"),
    Ticket("My dashboard shows a 500 error after the latest update.", "Technical"),
    Ticket("How do I change the email address on my account?", "Account"),
    Ticket("Do you have a mobile app, or is it web-only?", "General"),
    Ticket("The API rate limit seems too low for our usage — can it be increased?", "Technical"),
    Ticket("I want to cancel my subscription but keep my data. Is that possible?", "Account"),
    Ticket("Your product looks great! Can I schedule a demo for my team?", "General"),
    Ticket("The invoice I received doesn't match the price on your website.", "Billing"),
    Ticket("Integration with Salesforce stopped working after your last release.", "Technical"),
    Ticket("Can I add another admin user to my organization's account?", "Account"),
    Ticket("Is there a free trial available before I commit?", "Billing"),
    Ticket("The export to CSV feature is producing garbled characters for non-English text.", "Technical"),
    Ticket("I just wanted to say your support team is amazing — keep it up!", "General"),
    Ticket("How do I enable two-factor authentication for my team?", "Account"),
    Ticket("My credit card on file expired. Where do I update payment info?", "Billing"),
    Ticket("The search function returns zero results even when I know the item exists.", "Technical"),
    Ticket("Can you tell me about your data privacy and GDPR compliance?", "General"),
]

print(f"📊 Test set: {len(TEST_SET)} tickets")
label_counts = {}
for t in TEST_SET:
    label_counts[t.expected] = label_counts.get(t.expected, 0) + 1
for label in CATEGORIES:
    print(f"  {label:12s}: {label_counts.get(label, 0)}")

## Few-Shot Examples

6 hand-picked examples covering all four categories. We use the **first N**
examples for each experiment run (0, 2, 4, or 6 shots).

In [ ]:
@dataclass
class Example:
    """A labeled few-shot example."""
    text: str
    label: str


SHOT_EXAMPLES: list[Example] = [
    Example("I see a charge of $49.99 but I'm on the $29.99 plan. Please fix.", "Billing"),
    Example("The page loads indefinitely with a spinner after I log in.", "Technical"),
    Example("I need to reset my password but the link expired.", "Account"),
    Example("Where can I find documentation for your REST API?", "General"),
    Example("My annual renewal is coming up — any loyalty discounts?", "Billing"),
    Example("The Slack notification integration sends duplicate messages.", "Technical"),
]

for i, ex in enumerate(SHOT_EXAMPLES):
    print(f"  [{ex.label:11s}] {ex.text}")

## Prompt Builder

Uses `PromptTemplate` to construct a prompt that:
- Accepts `n_shots` — the number of few-shot examples to include
- Inserts those examples in a consistent format: `Example:\n<ticket>\nClassification: <label>`
- Appends the test ticket for classification
- Requests a single-word label as output

In [ ]:
def build_prompt(n_shots: int, ticket_text: str) -> str:
    """Build a classification prompt with n_shots few-shot examples.

    Uses PromptTemplate to format the system instruction, then manually
    appends examples and the query ticket (the template handles formatting;
    examples are procedural).
    """
    system_header = PromptTemplate(
        template_str=(
            "You are a customer support ticket classifier. "
            "Classify each ticket into exactly one category: "
            "Billing, Technical, Account, or General.\n"
            "Reply with ONLY the category name — no explanation.\n\n"
        ),
        version=1,
        metadata={"task": "ticket_classification"},
    )

    prompt_parts = [system_header.render()]

    # Add few-shot examples
    for i in range(min(n_shots, len(SHOT_EXAMPLES))):
        ex = SHOT_EXAMPLES[i]
        prompt_parts.append(f"Example:\n{ex.text}\nClassification: {ex.label}\n\n")

    # Use PromptTemplate for the query portion
    query_template = PromptTemplate(
        template_str="Ticket:\n{{ ticket }}\n\nClassification:",
        version=1,
    )
    prompt_parts.append(query_template.render(ticket=ticket_text))

    return "".join(prompt_parts)


# Show an example prompt with 2 shots
sample = build_prompt(2, "I was charged twice this month.")
print("=== Prompt (2-shot) ===")
print(sample)

## Experiment Runner

For each `n_shots ∈ {0, 2, 4, 6}`, we:
1. Build prompts for all 20 test tickets
2. Call the LLM (with `try/except` for robustness)
3. Extract the predicted label and compare against ground truth
4. Record per-shot accuracy

In [ ]:
import time


def classify(client, prompt: str) -> str:
    """Call the LLM and extract a clean label."""
    config = GenerationConfig.code()  # low temperature for deterministic output
    try:
        response = client.chat_completion(
            messages=[{"role": "user", "content": prompt}],
            **config.to_dict(),
        )
        raw = response["content"].strip()
    except Exception as exc:
        return f"[ERROR: {exc}]"

    # Normalize: extract the first matching category
    for cat in CATEGORIES:
        if cat.lower() in raw.lower():
            return cat
    # Fallback: return raw (may not match any category)
    return raw[:30]


def run_fewshot_experiment(n_shots: int, tickets: list[Ticket], client) -> dict:
    """Run the few-shot experiment for a given shot count.

    Returns a dict with per-ticket results and aggregate accuracy.
    """
    results = []
    correct = 0

    for i, ticket in enumerate(tickets):
        prompt = build_prompt(n_shots, ticket.text)
        predicted = classify(client, prompt)
        is_correct = predicted == ticket.expected
        if is_correct:
            correct += 1

        results.append({
            "index": i,
            "text": ticket.text[:60] + "...",
            "expected": ticket.expected,
            "predicted": predicted,
            "correct": is_correct,
        })

    accuracy = correct / len(tickets) * 100
    print(f"  {n_shots}-shot: {correct}/{len(tickets)} correct ({accuracy:.1f}%)")

    return {
        "n_shots": n_shots,
        "accuracy": accuracy,
        "correct": correct,
        "total": len(tickets),
        "results": results,
    }


# Run for 0, 2, 4, 6 shots
SHOT_COUNTS = [0, 2, 4, 6]
all_results: dict[int, dict] = {}

print("▶️  Running few-shot experiments...\n")
for n in SHOT_COUNTS:
    all_results[n] = run_fewshot_experiment(n, TEST_SET, client)
    if LIVE_API:
        time.sleep(0.5)  # rate-limit courtesy delay for real API

## Detailed Results Table

Per-ticket predictions across all shot counts, showing which tickets benefit
most from additional examples.

In [ ]:
from IPython.display import display, Markdown, HTML

# Build per-ticket table
rows = []
for i, ticket in enumerate(TEST_SET):
    row = {
        "#": i + 1,
        "Ticket": ticket.text[:55] + "..." if len(ticket.text) > 55 else ticket.text,
        "Expected": ticket.expected,
    }
    for n in SHOT_COUNTS:
        pred = all_results[n]["results"][i]["predicted"]
        is_correct = all_results[n]["results"][i]["correct"]
        row[f"{n}-shot"] = f"{'✅' if is_correct else '❌'} {pred}"
    rows.append(row)

try:
    import pandas as pd
    df = pd.DataFrame(rows)
    display(df)
except ImportError:
    print("⚠️  pandas not installed — displaying text table")
    for r in rows:
        shots_str = " | ".join(r[f"{n}-shot"] for n in SHOT_COUNTS)
        print(f"  #{r['#']:2d} [{r['Expected']:11s}] | {shots_str}")

## Accuracy Summary

Aggregate accuracy per shot count, including the marginal gain over the
previous shot level and the **cumulative gain vs 0-shot**.

In [ ]:
display(Markdown("### Accuracy vs Number of Shots\n"))

zero_shot_acc = all_results[0]["accuracy"]

print(f"{'Shots':<8} {'Accuracy':<12} {'Δ vs prev':<12} {'Δ vs 0-shot':<12}")
print("-" * 44)
prev_acc = None
for n in SHOT_COUNTS:
    acc = all_results[n]["accuracy"]
    delta_prev = "" if prev_acc is None else f"+{acc - prev_acc:.1f} pp"
    delta_zero = f"+{acc - zero_shot_acc:.1f} pp" if n > 0 else "baseline"
    bar = "█" * int(acc / 5)
    print(f"{n:<8} {acc:5.1f}% {bar:<20s} {delta_prev:<12s} {delta_zero:<12s}")
    prev_acc = acc

## Visualization

Line chart showing accuracy improvement with increasing shots. The
diminishing-returns pattern is annotated.

In [ ]:
try:
    import matplotlib.pyplot as plt
    import numpy as np

    accuracies = [all_results[n]["accuracy"] for n in SHOT_COUNTS]
    gains = [accuracies[i] - accuracies[i - 1] for i in range(1, len(accuracies))]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Chart 1: Accuracy vs shots
    axes[0].plot(SHOT_COUNTS, accuracies, "o-", color="#3498db", linewidth=2.5, markersize=10,
                 markerfacecolor="white", markeredgewidth=2)
    axes[0].fill_between(SHOT_COUNTS, 0, accuracies, alpha=0.08, color="#3498db")
    axes[0].set_xlabel("Number of Few-Shot Examples")
    axes[0].set_ylabel("Accuracy (%)")
    axes[0].set_title("Classification Accuracy vs Few-Shot Examples")
    axes[0].set_xticks(SHOT_COUNTS)
    axes[0].set_ylim(0, 105)
    axes[0].grid(True, alpha=0.3)
    for n, acc in zip(SHOT_COUNTS, accuracies):
        axes[0].annotate(f"{acc:.1f}%", (n, acc), textcoords="offset points",
                        xytext=(0, 12), ha="center", fontweight="bold", fontsize=11)

    # Chart 2: Marginal gain per additional 2 shots
    shot_labels = [f"0→2" , "2→4", "4→6"]
    colors = ["#2ecc71" if g > 0 else "#e74c3c" for g in gains]
    bars = axes[1].bar(shot_labels, gains, color=colors, alpha=0.85, width=0.5)
    axes[1].axhline(y=0, color="gray", linestyle="--", linewidth=0.8)
    axes[1].set_ylabel("Accuracy Gain (pp)")
    axes[1].set_title("Marginal Gain per +2 Shots (Diminishing Returns)")
    for bar, g in zip(bars, gains):
        y_pos = bar.get_height() + 0.3 if g >= 0 else bar.get_height() - 1.5
        axes[1].text(bar.get_x() + bar.get_width() / 2, y_pos,
                     f"{g:+.1f} pp", ha="center", fontweight="bold", fontsize=11)

    plt.tight_layout()
    plt.show()

except ImportError:
    print("⚠️  matplotlib not installed — skipping visualization.")
    print("    Install with: pip install matplotlib")

## Analysis

### Key Findings

1. **0-shot is the weakest.** Without any examples, the model relies entirely on
   its pretrained understanding of category boundaries. Ambiguous cases (e.g.,
   "What's the difference between plans?" — could be Billing or General) are
   often misclassified.

2. **2-shot provides a large jump.** Just two examples calibrate the output
   format (single-word label) and demonstrate the classification logic. This is
   the **highest marginal gain** — going from 0 to 2 shots typically yields
   +15–25 percentage points.

3. **4-shot refines edge cases.** Two additional examples cover a broader range
   of category nuances. The gain is smaller but still meaningful.

4. **6-shot shows diminishing returns.** The marginal gain from 4→6 shots is
   minimal — the model has already seen enough variety. Further examples add
   token cost without proportional accuracy benefit.

### Practical Takeaways

- **Start with 2–4 shots** — this is the sweet spot for most classification tasks.
- **Diversify your examples** — one per category is better than multiple from
  the same category.
- **Beyond ~8 shots, switch to fine-tuning** — in-context learning plateaus;
  gradient-based optimization scales better for very high accuracy targets.
- **Monitor token cost** — each additional example adds roughly the same token
  count to every inference call, which compounds at scale.

### Caveats (Mock Mode)

If running in mock mode, the accuracy curve is **synthetic** — designed to
illustrate the diminishing-returns pattern. The mock assigns each test case a
"difficulty threshold" that determines how many shots are needed to classify
it correctly. To replicate with a real LLM, set `OPENAI_API_KEY`.

In [ ]:
# Summary
print("=" * 50)
print("EXPERIMENT SUMMARY")
print("=" * 50)
print(f"Test cases:     {len(TEST_SET)}")
print(f"Categories:     {', '.join(CATEGORIES)}")
print(f"Shot counts:    {SHOT_COUNTS}")
print()
for n in SHOT_COUNTS:
    r = all_results[n]
    bar = "█" * int(r["accuracy"] / 5)
    print(f"  {n}-shot:  {r['correct']:2d}/{r['total']:2d}  {r['accuracy']:5.1f}%  {bar}")
print()
best_n = max(SHOT_COUNTS, key=lambda n: all_results[n]["accuracy"])
print(f"Best:           {best_n}-shot at {all_results[best_n]['accuracy']:.1f}%")
print(f"Client:         {client.model_name} {'(live)' if LIVE_API else '(mock)'}")